# MedFlow — 00 · Ingestão de dados (camada Bronze)

Este notebook **somente ingere e preserva** as fontes do DATASUS, Ministério
da Saúde e IBGE. Não aplica regra de negócio, de/para, filtro analítico,
imputação nem cálculo de indicador. Os únicos acréscimos são colunas técnicas
de linhagem e um manifesto com contagens, esquema e hashes.

**Entradas:** SIH/RD, CNES/LT, API de localidades do IBGE, API DEMAS de
regiões e estabelecimentos, CONCLA/IBGE e referência CID-10 do DATASUS.  
**Saídas:** `dados/bronze/`, mantendo `dados/raw/` como cache dos arquivos DBC/DBF.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from ftplib import FTP
from hashlib import sha256
from pathlib import Path
from datetime import datetime, timezone
import gzip
import json
import subprocess
import urllib.request
from zipfile import ZipFile

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import datasus_dbc
from dbfread import DBF

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_CACHE = BASE / "dados" / "raw"
DIR_BRONZE = BASE / "dados" / "bronze"
for pasta in (DIR_CACHE, DIR_BRONZE):
    pasta.mkdir(parents=True, exist_ok=True)

UF = "SP"
PERIODO_INICIAL = (2024, 1)
PERIODO_FINAL = (2026, 12)
FTP_HOST = "ftp.datasus.gov.br"
FTP_DIRS = {
    "RD": "/dissemin/publicos/SIHSUS/200801_/Dados",
    "LT": "/dissemin/publicos/CNES/200508_/Dados/LT",
}
SOBRESCREVER = False
ARQ_MANIFESTO = DIR_BRONZE / "MANIFESTO.json"
ARQ_SIH = DIR_BRONZE / "sih_rd_sp_2024_2026.parquet"
ARQ_CNES = DIR_BRONZE / "cnes_lt_sp_2024_2026.parquet"

def nome_arquivo(grupo, ano, mes):
    return f"{grupo}{UF}{str(ano)[2:]}{mes:02d}.dbc"

def listar_remotos(grupo):
    ftp = FTP(FTP_HOST, timeout=120)
    try:
        ftp.login()
        ftp.cwd(FTP_DIRS[grupo])
        return {Path(nome).name.upper(): Path(nome).name for nome in ftp.nlst()}
    finally:
        try:
            ftp.quit()
        except Exception:
            ftp.close()

def extrair_competencias(grupo, remotos):
    prefixo = f"{grupo}{UF}"
    competencias = set()
    for nome in remotos:
        if nome.startswith(prefixo) and nome.endswith(".DBC") and len(nome) == 12:
            ano, mes = 2000 + int(nome[4:6]), int(nome[6:8])
            if 1 <= mes <= 12 and PERIODO_INICIAL <= (ano, mes) <= PERIODO_FINAL:
                competencias.add((ano, mes))
    return competencias

LISTAGENS_REMOTAS = {grupo: listar_remotos(grupo) for grupo in ("RD", "LT")}
DISPONIVEIS = {
    grupo: extrair_competencias(grupo, remotos)
    for grupo, remotos in LISTAGENS_REMOTAS.items()
}
COMPETENCIAS = sorted(DISPONIVEIS["RD"] & DISPONIVEIS["LT"])
assert COMPETENCIAS and COMPETENCIAS[0] == PERIODO_INICIAL, (
    f"recorte não começa em {PERIODO_INICIAL}: {COMPETENCIAS[:3]}"
)
esperadas = []
ano, mes = PERIODO_INICIAL
while (ano, mes) <= COMPETENCIAS[-1]:
    esperadas.append((ano, mes))
    ano, mes = (ano + 1, 1) if mes == 12 else (ano, mes + 1)
assert COMPETENCIAS == esperadas, "há lacuna no intervalo comum entre SIH/RD e CNES/LT"

competencias_anteriores = []
if ARQ_MANIFESTO.exists():
    manifesto_anterior = json.loads(ARQ_MANIFESTO.read_text(encoding="utf-8"))
    competencias_anteriores = manifesto_anterior.get("recorte", {}).get("competencias", [])
competencias_atuais = [f"{a}{m:02d}" for a, m in COMPETENCIAS]
RECORTE_MUDOU = competencias_anteriores != competencias_atuais

print("cache :", DIR_CACHE)
print("bronze:", DIR_BRONZE)
print("recorte:", UF, COMPETENCIAS[0], "a", COMPETENCIAS[-1])
print("competências comuns:", len(COMPETENCIAS))
print("última RD:", max(DISPONIVEIS["RD"]), "| última LT:", max(DISPONIVEIS["LT"]))

## 1. Download das fontes

O download é idempotente: arquivos existentes no cache não são substituídos.
Um arquivo parcial só recebe o nome definitivo quando o download termina.

In [ ]:
def baixar_grupo(grupo):
    alvos = [nome_arquivo(grupo, ano, mes) for ano, mes in COMPETENCIAS]
    faltantes = [nome for nome in alvos if not (DIR_CACHE / nome).exists()]
    if not faltantes:
        print(f"[{grupo}] {len(alvos)} arquivos no cache")
        return
    ftp = FTP(FTP_HOST, timeout=120)
    try:
        ftp.login()
        ftp.cwd(FTP_DIRS[grupo])
        remotos = LISTAGENS_REMOTAS[grupo]
        for nome in faltantes:
            real = remotos.get(nome.upper())
            assert real, f"arquivo ausente no FTP: {nome}"
            destino = DIR_CACHE / nome
            parcial = destino.with_suffix(".dbc.parcial")
            with parcial.open("wb") as arquivo:
                ftp.retrbinary(f"RETR {real}", arquivo.write)
            parcial.rename(destino)
            print("baixado:", nome)
    finally:
        try:
            ftp.quit()
        except Exception:
            ftp.close()

for grupo in ("RD", "LT"):
    baixar_grupo(grupo)

## 2. Descompressão DBC → DBF

O DBF é uma representação técnica do arquivo recebido. Nenhum registro ou
campo é alterado nesta etapa.

In [ ]:
for grupo in ("RD", "LT"):
    for ano, mes in COMPETENCIAS:
        dbc = DIR_CACHE / nome_arquivo(grupo, ano, mes)
        dbf = dbc.with_suffix(".dbf")
        if not dbf.exists():
            datasus_dbc.decompress(str(dbc), str(dbf))
            print("convertido:", dbf.name)

## 3. Serialização fiel em Parquet

Os valores entregues pelo leitor DBF são mantidos. `_arquivo_fonte`,
`_ano_arquivo` e `_mes_arquivo` são metadados de linhagem, derivados do nome
do arquivo — não são atributos clínicos.

In [ ]:
MUDANCAS_ESQUEMA = {}

def ler_competencia(grupo, ano, mes):
    dbf = (DIR_CACHE / nome_arquivo(grupo, ano, mes)).with_suffix(".dbf")
    frame = pd.DataFrame(iter(DBF(str(dbf), encoding="iso-8859-1")))
    frame["_arquivo_fonte"] = dbf.name
    frame["_ano_arquivo"] = ano
    frame["_mes_arquivo"] = mes
    return frame

def consolidar(grupo, nome_saida):
    destino = DIR_BRONZE / nome_saida
    campos_por_competencia = {
        f"{ano}{mes:02d}": set(
            DBF(
                str((DIR_CACHE / nome_arquivo(grupo, ano, mes)).with_suffix(".dbf")),
                encoding="iso-8859-1", load=False,
            ).field_names
        ) | {"_arquivo_fonte", "_ano_arquivo", "_mes_arquivo"}
        for ano, mes in COMPETENCIAS
    }
    campos_iniciais = campos_por_competencia[competencias_atuais[0]]
    campos_uniao = set().union(*campos_por_competencia.values())
    MUDANCAS_ESQUEMA[grupo] = {
        competencia: {
            "adicionadas_desde_inicio": sorted(campos - campos_iniciais),
            "ausentes_em_relacao_uniao": sorted(campos_uniao - campos),
        }
        for competencia, campos in campos_por_competencia.items()
        if campos != campos_iniciais
    }
    if MUDANCAS_ESQUEMA[grupo]:
        print(f"[{grupo}] evolução de esquema:", MUDANCAS_ESQUEMA[grupo])

    if destino.exists() and not SOBRESCREVER:
        cobertura = pd.read_parquet(
            destino, columns=["_ano_arquivo", "_mes_arquivo"]
        ).drop_duplicates()
        competencias_destino = set(map(tuple, cobertura.to_numpy()))
        if competencias_destino == set(COMPETENCIAS):
            print("já existe com o recorte atual; validaremos sem substituir:", destino.name)
            return

    temporario = destino.with_suffix(".parquet.parcial")
    if temporario.exists():
        temporario.unlink()
    esquemas = []
    for ano, mes in COMPETENCIAS:
        frame = ler_competencia(grupo, ano, mes)
        tabela = pa.Table.from_pandas(frame, preserve_index=False)
        esquemas.append(tabela.schema.remove_metadata())
    esquema_union = pa.unify_schemas(esquemas)

    writer = None
    total = 0
    try:
        for ano, mes in COMPETENCIAS:
            frame = ler_competencia(grupo, ano, mes)
            tabela = pa.Table.from_pandas(frame, preserve_index=False)
            for campo in esquema_union:
                if campo.name not in tabela.column_names:
                    tabela = tabela.append_column(
                        campo, pa.nulls(len(tabela), type=campo.type)
                    )
            tabela = tabela.select(esquema_union.names).cast(esquema_union)
            if writer is None:
                writer = pq.ParquetWriter(temporario, esquema_union, compression="snappy")
            writer.write_table(tabela)
            total += len(frame)
            print(f"[{grupo}] {ano}-{mes:02d}: {len(frame):,} | acumulado {total:,}")
    finally:
        if writer is not None:
            writer.close()
    temporario.replace(destino)

consolidar("RD", ARQ_SIH.name)
consolidar("LT", ARQ_CNES.name)

## 4. Referência bruta do IBGE

A resposta JSON é salva byte a byte. A transformação para dimensão municipal
pertence à Silver.

In [ ]:
URL_IBGE = "https://servicodados.ibge.gov.br/api/v1/localidades/estados/35/municipios"
ARQ_IBGE = DIR_BRONZE / "ibge_municipios_sp_raw.json"
if not ARQ_IBGE.exists() or SOBRESCREVER:
    with urllib.request.urlopen(URL_IBGE, timeout=60) as resposta:
        conteudo_ibge = resposta.read()
    ARQ_IBGE.write_bytes(conteudo_ibge)
else:
    conteudo_ibge = ARQ_IBGE.read_bytes()
if conteudo_ibge.startswith(b"\x1f\x8b"):
    conteudo_ibge = gzip.decompress(conteudo_ibge)
    ARQ_IBGE.write_bytes(conteudo_ibge)
print("IBGE:", len(json.loads(conteudo_ibge)), "registros")

## 5. Referências cadastrais e terminológicas oficiais

As respostas são preservadas na Bronze. O cadastro de estabelecimentos é uma
fotografia **atual** consultada por CNES e será identificado como enriquecimento
não histórico na Silver. Região de saúde é obtida pela API oficial por
município; CID-10 vem do pacote oficial do DATASUS; natureza jurídica é
preservada a partir da página CONCLA/IBGE.

In [ ]:
def baixar_referencia(url, destino, timeout=180):
    if destino.exists() and not SOBRESCREVER:
        return destino.read_bytes()
    requisicao = urllib.request.Request(
        url, headers={"User-Agent": "MedFlow-FIAP/1.0", "Accept-Encoding": "identity"}
    )
    with urllib.request.urlopen(requisicao, timeout=timeout) as resposta:
        conteudo = resposta.read()
    if conteudo.startswith(b"\x1f\x8b"):
        conteudo = gzip.decompress(conteudo)
    destino.write_bytes(conteudo)
    return conteudo


URL_REGIOES = (
    "https://apidadosabertos.saude.gov.br/"
    "macrorregiao-e-regiao-de-saude/municipio?sigla_uf=SP&limit=860&offset=0"
)
ARQ_REGIOES = DIR_BRONZE / "ms_regioes_saude_sp_raw.json"
regioes_bytes = baixar_referencia(URL_REGIOES, ARQ_REGIOES)
regioes_payload = json.loads(regioes_bytes)
regioes = regioes_payload["macrorregiao_regiao_saude_municipios"]

URL_CID10 = "http://www2.datasus.gov.br/cid10/V2008/downloads/CID10CSV.zip"
ARQ_CID10 = DIR_BRONZE / "datasus_cid10_2008.zip"
# O servidor legado da CID-10 expira via urllib, mas responde via curl.
if not ARQ_CID10.exists() or SOBRESCREVER:
    subprocess.run([
        "curl", "-L", "--max-time", "120", "-sS", URL_CID10,
        "-o", str(ARQ_CID10),
    ], check=True)
with ZipFile(ARQ_CID10) as pacote:
    arquivos_cid = pacote.namelist()

URL_CONCLA = (
    "https://concla.ibge.gov.br/documentacao/3051-concla/estrutura/"
    "natureza-juridica-2021.html"
)
ARQ_CONCLA = DIR_BRONZE / "ibge_concla_natureza_juridica_2021.html"
baixar_referencia(URL_CONCLA, ARQ_CONCLA)

URL_CNES_MODELO = "https://apidadosabertos.saude.gov.br/cnes/estabelecimentos/{cnes}"
ARQ_CNES_ATUAL = DIR_BRONZE / "ms_cnes_estabelecimentos_atuais_raw.json"

codigos_cnes = sorted(
    pd.read_parquet(ARQ_SIH, columns=["CNES"])
    .CNES.astype("string").str.strip().str.zfill(7).unique()
)
registros_cache = {}
if ARQ_CNES_ATUAL.exists():
    payload_cache = json.loads(ARQ_CNES_ATUAL.read_bytes())
    registros_cache = {
        str(item["codigo_cnes"]).replace(".0", "").zfill(7): item
        for item in payload_cache.get("registros", [])
    }

def consultar_cnes(codigo):
    ultimo_erro = None
    for tentativa in range(3):
        try:
            requisicao = urllib.request.Request(
                URL_CNES_MODELO.format(cnes=codigo),
                headers={"User-Agent": "MedFlow-FIAP/1.0"},
            )
            with urllib.request.urlopen(requisicao, timeout=30) as resposta:
                return codigo, json.loads(resposta.read())
        except Exception as erro:
            ultimo_erro = erro
    return codigo, {"_erro": str(ultimo_erro)}

faltantes_cnes = codigos_cnes if SOBRESCREVER else sorted(set(codigos_cnes) - set(registros_cache))
respostas = {} if SOBRESCREVER else dict(registros_cache)
if faltantes_cnes:
    print("estabelecimentos CNES a consultar:", len(faltantes_cnes))
    with ThreadPoolExecutor(max_workers=12) as executor:
        futuros = [executor.submit(consultar_cnes, codigo) for codigo in faltantes_cnes]
        for futuro in as_completed(futuros):
            codigo, resposta = futuro.result()
            respostas[codigo] = resposta

falhas = {codigo: item for codigo, item in respostas.items() if "_erro" in item}
assert not falhas, f"falhas na API CNES: {list(falhas.items())[:10]}"
cnes_atual_payload = {
    "fonte": URL_CNES_MODELO,
    "extraido_em_utc": datetime.now(timezone.utc).isoformat(),
    "observacao": "cadastro atual; usar somente como enriquecimento não histórico",
    "registros": [respostas[codigo] for codigo in codigos_cnes],
}
if faltantes_cnes or SOBRESCREVER or not ARQ_CNES_ATUAL.exists():
    ARQ_CNES_ATUAL.write_text(
        json.dumps(cnes_atual_payload, ensure_ascii=False, indent=2), encoding="utf-8"
    )

print("regiões/municípios MS:", len(regioes))
print("arquivos no pacote CID-10:", len(arquivos_cid))
print("estabelecimentos CNES atuais:", len(cnes_atual_payload["registros"]))
print("CONCLA natureza jurídica:", ARQ_CONCLA.stat().st_size, "bytes")

## 6. Manifesto e validação da Bronze

Esta validação responde apenas se a ingestão está completa e reproduzível.
Validade semântica dos domínios é responsabilidade do notebook Silver.

In [ ]:
def hash_arquivo(caminho, bloco=1024 * 1024):
    digest = sha256()
    with caminho.open("rb") as arquivo:
        for parte in iter(lambda: arquivo.read(bloco), b""):
            digest.update(parte)
    return digest.hexdigest()

arquivos = {
    "sih": ARQ_SIH,
    "cnes": ARQ_CNES,
    "ibge": ARQ_IBGE,
    "regioes_saude_ms": ARQ_REGIOES,
    "cid10_datasus": ARQ_CID10,
    "natureza_juridica_concla": ARQ_CONCLA,
    "cnes_estabelecimentos_atuais": ARQ_CNES_ATUAL,
}
sih_meta = pq.ParquetFile(arquivos["sih"])
cnes_meta = pq.ParquetFile(arquivos["cnes"])
ibge_qtd = len(json.loads(ARQ_IBGE.read_bytes()))

checks = {
    "arquivos_rd_cache": sum((DIR_CACHE / nome_arquivo("RD", a, m)).exists() for a, m in COMPETENCIAS),
    "arquivos_lt_cache": sum((DIR_CACHE / nome_arquivo("LT", a, m)).exists() for a, m in COMPETENCIAS),
    "linhas_sih": sih_meta.metadata.num_rows,
    "colunas_sih": len(sih_meta.schema_arrow.names),
    "linhas_cnes": cnes_meta.metadata.num_rows,
    "colunas_cnes": len(cnes_meta.schema_arrow.names),
    "municipios_ibge": ibge_qtd,
    "municipios_regiao_saude_ms": len(regioes),
    "estabelecimentos_cnes_atuais": len(cnes_atual_payload["registros"]),
    "arquivos_pacote_cid10": len(arquivos_cid),
}
validacoes = {
    "arquivos_rd_cache": checks["arquivos_rd_cache"] == len(COMPETENCIAS),
    "arquivos_lt_cache": checks["arquivos_lt_cache"] == len(COMPETENCIAS),
    "linhas_sih": checks["linhas_sih"] > 0,
    "colunas_sih": checks["colunas_sih"] >= 116,
    "linhas_cnes": checks["linhas_cnes"] > 0,
    "colunas_cnes": checks["colunas_cnes"] == 31,
    "municipios_ibge": checks["municipios_ibge"] == 645,
    "municipios_regiao_saude_ms": checks["municipios_regiao_saude_ms"] == 645,
    "estabelecimentos_cnes_atuais": checks["estabelecimentos_cnes_atuais"] == len(codigos_cnes),
    "arquivos_pacote_cid10": checks["arquivos_pacote_cid10"] == 6,
}
for chave, valor in checks.items():
    print(f"{chave:<32} {valor:>12,} | {'OK' if validacoes[chave] else 'FALHOU'}")
assert all(validacoes.values()), {k: checks[k] for k, ok in validacoes.items() if not ok}

manifesto = {
    "camada": "bronze",
    "gerado_em_utc": datetime.now(timezone.utc).isoformat(),
    "recorte": {
        "uf": UF,
        "periodo_solicitado": [f"{PERIODO_INICIAL[0]}{PERIODO_INICIAL[1]:02d}", f"{PERIODO_FINAL[0]}{PERIODO_FINAL[1]:02d}"],
        "ultima_competencia_comum": competencias_atuais[-1],
        "competencias": competencias_atuais,
    },
    "atualizacao": "descoberta remota; download incremental; consolidação promovida por arquivo parcial",
    "evolucao_esquema": MUDANCAS_ESQUEMA,
    "principio": "fontes preservadas; sem regra de negócio, de/para ou filtro analítico",
    "fontes": {
        "SIH_RD": FTP_DIRS["RD"],
        "CNES_LT": FTP_DIRS["LT"],
        "IBGE_municipios": URL_IBGE,
        "MS_DEMAS_regioes_saude": URL_REGIOES,
        "MS_DEMAS_cnes_atual": URL_CNES_MODELO,
        "DATASUS_CID10": URL_CID10,
        "IBGE_CONCLA_natureza_juridica": URL_CONCLA,
    },
    "checks": checks,
    "arquivos": {
        nome: {
            "caminho": caminho.name,
            "bytes": caminho.stat().st_size,
            "sha256": hash_arquivo(caminho),
        }
        for nome, caminho in arquivos.items()
    },
}
ARQ_MANIFESTO.write_text(
    json.dumps(manifesto, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("\nBRONZE VÁLIDA — ingestão completa; seguir para o notebook 01.")